# Understanding Data Preparation in nanoGPT

This notebook explains how `prepare.py` transforms raw text into `train.bin`.

Key concepts:
- HuggingFace `Dataset` objects (dict-like but with methods)
- `.map()` for tokenization
- `.shard()` for batch processing
- Writing to binary files with `np.memmap`

In [1]:
import numpy as np
from datasets import Dataset
import tiktoken

enc = tiktoken.get_encoding("gpt2")
print(f"GPT-2 vocab size: {enc.n_vocab}")
print(f"End-of-text token: {enc.eot_token} = '{enc.decode([enc.eot_token])!r}'")

GPT-2 vocab size: 50257
End-of-text token: 50256 = ''<|endoftext|>''


## 1. HuggingFace Dataset: Not a dict, but acts like one

A `Dataset` is a special object that stores tabular data (like a DataFrame).
You can access columns like a dict, but it also has methods like `.map()` and `.shard()`.

In [2]:
# Create a simple Dataset from a list of dicts
documents = [
    {"text": "Hello world!"},
    {"text": "How are you today?"},
    {"text": "Machine learning is fun."},
    {"text": "GPT learns from text."},
    {"text": "This is the last document."},
]

dataset = Dataset.from_list(documents)

print("Type:", type(dataset))
print("Length:", len(dataset))
print("\nDataset object:")
print(dataset)

Type: <class 'datasets.arrow_dataset.Dataset'>
Length: 5

Dataset object:
Dataset({
    features: ['text'],
    num_rows: 5
})


In [3]:
# Dict-like access to columns
print("Access column like dict:")
print(f"  dataset['text'] = {dataset['text']}")

print("\nAccess single row by index:")
print(f"  dataset[0] = {dataset[0]}")
print(f"  dataset[2] = {dataset[2]}")

print("\nAccess slice:")
print(f"  dataset[1:3] = {dataset[1:3]}")

Access column like dict:
  dataset['text'] = Column(['Hello world!', 'How are you today?', 'Machine learning is fun.', 'GPT learns from text.', 'This is the last document.'])

Access single row by index:
  dataset[0] = {'text': 'Hello world!'}
  dataset[2] = {'text': 'Machine learning is fun.'}

Access slice:
  dataset[1:3] = {'text': ['How are you today?', 'Machine learning is fun.']}


## 2. `.map()` - Apply function to each row

This is how tokenization happens. The function receives one row (a dict) and returns a new dict.

In [4]:
# This is exactly what prepare.py does
def process(example):
    """
    Input:  {'text': 'Hello world!'}
    Output: {'ids': [15496, 995, 0, 50256], 'len': 4}
    """
    ids = enc.encode_ordinary(example['text'])  # Tokenize
    ids.append(enc.eot_token)                    # Add end-of-text token
    return {'ids': ids, 'len': len(ids)}

# Apply to each row
tokenized = dataset.map(process, remove_columns=['text'])

print("After .map():")
print(tokenized)
print("\nColumns changed: 'text' removed, 'ids' and 'len' added")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

After .map():
Dataset({
    features: ['ids', 'len'],
    num_rows: 5
})

Columns changed: 'text' removed, 'ids' and 'len' added


In [5]:
# Look at the tokenized data
print("Tokenized dataset row by row:\n")
for i in range(len(tokenized)):
    row = tokenized[i]
    ids = row['ids']
    text = enc.decode(ids[:-1])  # Decode without EOT for readability
    print(f"Row {i}:")
    print(f"  Original: '{text}'")
    print(f"  ids: {ids}")
    print(f"  len: {row['len']}")
    print(f"  Last token {ids[-1]} = EOT")
    print()

Tokenized dataset row by row:

Row 0:
  Original: 'Hello world!'
  ids: [15496, 995, 0, 50256]
  len: 4
  Last token 50256 = EOT

Row 1:
  Original: 'How are you today?'
  ids: [2437, 389, 345, 1909, 30, 50256]
  len: 6
  Last token 50256 = EOT

Row 2:
  Original: 'Machine learning is fun.'
  ids: [37573, 4673, 318, 1257, 13, 50256]
  len: 6
  Last token 50256 = EOT

Row 3:
  Original: 'GPT learns from text.'
  ids: [38, 11571, 22974, 422, 2420, 13, 50256]
  len: 7
  Last token 50256 = EOT

Row 4:
  Original: 'This is the last document.'
  ids: [1212, 318, 262, 938, 3188, 13, 50256]
  len: 7
  Last token 50256 = EOT



## 3. `.shard()` - Split dataset into chunks

Used to process large datasets in batches without loading everything into memory.

In [6]:
# Split into 2 shards (in real code: 1024 shards)
print("Original dataset has 5 rows")
print(f"  tokenized['ids'] = {tokenized['ids']}\n")

# Get shard 0 (first half)
shard0 = tokenized.shard(num_shards=2, index=0, contiguous=True)
print("Shard 0 (first half):")
print(f"  Length: {len(shard0)}")
print(f"  ids: {shard0['ids']}")

# Get shard 1 (second half)
shard1 = tokenized.shard(num_shards=2, index=1, contiguous=True)
print("\nShard 1 (second half):")
print(f"  Length: {len(shard1)}")
print(f"  ids: {shard1['ids']}")

Original dataset has 5 rows
  tokenized['ids'] = Column([[15496, 995, 0, 50256], [2437, 389, 345, 1909, 30, 50256], [37573, 4673, 318, 1257, 13, 50256], [38, 11571, 22974, 422, 2420, 13, 50256], [1212, 318, 262, 938, 3188, 13, 50256]])

Shard 0 (first half):
  Length: 3
  ids: Column([[15496, 995, 0, 50256], [2437, 389, 345, 1909, 30, 50256], [37573, 4673, 318, 1257, 13, 50256]])

Shard 1 (second half):
  Length: 2
  ids: Column([[38, 11571, 22974, 422, 2420, 13, 50256], [1212, 318, 262, 938, 3188, 13, 50256]])


## 4. `np.concatenate()` - Flatten all documents into one array

Each shard contains multiple documents. We flatten them into a single array.

In [7]:
# Convert to numpy format (required for concatenate)
shard0_np = shard0.with_format('numpy')

print("shard0['ids'] is a list of arrays (one per document):")
for i, ids in enumerate(shard0_np['ids']):
    print(f"  Doc {i}: {ids} (type: {type(ids).__name__})")

print("\nnp.concatenate() flattens into single array:")
flat = np.concatenate(shard0_np['ids'])
print(f"  Result: {flat}")
print(f"  Shape: {flat.shape}")
print(f"  Type: {flat.dtype}")

shard0['ids'] is a list of arrays (one per document):
  Doc 0: [15496   995     0 50256] (type: ndarray)
  Doc 1: [ 2437   389   345  1909    30 50256] (type: ndarray)
  Doc 2: [37573  4673   318  1257    13 50256] (type: ndarray)

np.concatenate() flattens into single array:
  Result: [15496   995     0 50256  2437   389   345  1909    30 50256 37573  4673
   318  1257    13 50256]
  Shape: (16,)
  Type: int64


## 5. Full Pipeline: Text → train.bin

Putting it all together - this is what `prepare.py` does.

In [8]:
# Step 1: Calculate total length
total_tokens = np.sum(tokenized['len'], dtype=np.uint64)
print(f"Step 1: Total tokens across all documents: {total_tokens}")

# Step 2: Create output array (in real code: memory-mapped file)
output = np.zeros(total_tokens, dtype=np.uint16)
print(f"Step 2: Created output array of shape {output.shape}")

# Step 3: Process in shards and write
num_shards = 2
idx = 0

print(f"\nStep 3: Process {num_shards} shards:")
for shard_idx in range(num_shards):
    # Get shard
    shard = tokenized.shard(num_shards=num_shards, index=shard_idx, contiguous=True)
    shard = shard.with_format('numpy')
    
    # Concatenate all documents in this shard
    arr_batch = np.concatenate(shard['ids'])
    
    # Write to output array
    output[idx:idx + len(arr_batch)] = arr_batch
    print(f"  Shard {shard_idx}: {len(shard)} docs, {len(arr_batch)} tokens → output[{idx}:{idx+len(arr_batch)}]")
    
    idx += len(arr_batch)

print(f"\nFinal output array: {output}")
print(f"This is what train.bin contains!")

Step 1: Total tokens across all documents: 30
Step 2: Created output array of shape (30,)

Step 3: Process 2 shards:
  Shard 0: 3 docs, 16 tokens → output[0:16]
  Shard 1: 2 docs, 14 tokens → output[16:30]

Final output array: [15496   995     0 50256  2437   389   345  1909    30 50256 37573  4673
   318  1257    13 50256    38 11571 22974   422  2420    13 50256  1212
   318   262   938  3188    13 50256]
This is what train.bin contains!


In [9]:
# Visualize the structure - where are the EOT tokens?
print("Visualizing train.bin structure:\n")
print("Token ID | Decoded")
print("-" * 40)

for i, token in enumerate(output):
    decoded = enc.decode([token])
    if token == enc.eot_token:
        print(f"{i:3}: {token:5} | <|endoftext|> ← Document boundary!")
    else:
        print(f"{i:3}: {token:5} | {decoded!r}")

print("\n" + "=" * 50)
print("Notice: EOT tokens (50256) mark where documents end.")
print("The model learns these boundaries during training.")

Visualizing train.bin structure:

Token ID | Decoded
----------------------------------------
  0: 15496 | 'Hello'
  1:   995 | ' world'
  2:     0 | '!'
  3: 50256 | <|endoftext|> ← Document boundary!
  4:  2437 | 'How'
  5:   389 | ' are'
  6:   345 | ' you'
  7:  1909 | ' today'
  8:    30 | '?'
  9: 50256 | <|endoftext|> ← Document boundary!
 10: 37573 | 'Machine'
 11:  4673 | ' learning'
 12:   318 | ' is'
 13:  1257 | ' fun'
 14:    13 | '.'
 15: 50256 | <|endoftext|> ← Document boundary!
 16:    38 | 'G'
 17: 11571 | 'PT'
 18: 22974 | ' learns'
 19:   422 | ' from'
 20:  2420 | ' text'
 21:    13 | '.'
 22: 50256 | <|endoftext|> ← Document boundary!
 23:  1212 | 'This'
 24:   318 | ' is'
 25:   262 | ' the'
 26:   938 | ' last'
 27:  3188 | ' document'
 28:    13 | '.'
 29: 50256 | <|endoftext|> ← Document boundary!

Notice: EOT tokens (50256) mark where documents end.
The model learns these boundaries during training.


## Summary

```
Raw Text Documents
       │
       ▼ .map(process)
┌──────────────────────────────────────────┐
│ Dataset with 'ids' and 'len' columns     │
│   Row 0: ids=[15496, 995, 50256], len=3  │
│   Row 1: ids=[2437, 389, 50256], len=3   │
│   ...                                     │
└──────────────────────────────────────────┘
       │
       ▼ .shard() + np.concatenate()
┌──────────────────────────────────────────┐
│ Flat 1D array: [15496, 995, 50256, 2437, │
│                 389, 50256, ...]         │
└──────────────────────────────────────────┘
       │
       ▼ np.memmap (write to disk)
┌──────────────────────────────────────────┐
│ train.bin (binary file on disk)          │
│ Just raw bytes: 2 bytes per token        │
└──────────────────────────────────────────┘
```

**Key insight:** Documents are separated by EOT token (50256), not by file structure.